In [17]:
import ast
import logging
import os

import json
import groq
import pandas as pd
from groq import Groq
from dotenv import load_dotenv

In [19]:
load_dotenv()
client = Groq(api_key=os.getenv("GROQ_API_KEY"))

In [20]:
df_train = pd.read_csv("./train.csv")
df_test = pd.read_csv("./test.csv")
df = pd.concat([df_train, df_test])

In [21]:
# Variant A
prompt_v1 = """
Extract features from the text enclosed between triple backquotes.
Return ONLY a valid JSON dictionary with double quotes, no extra text.
Keys must remain unchanged.
If a feature value is mentioned, replace 'None' with it.
If not mentioned, keep 'None'.
Infer values when possible, but only from allowed categories.
"""

# Variant B (stricter formatting + inference emphasis)
prompt_v2 = """
You are an information extraction system.
Task: From the text between triple backquotes, fill the features dictionary.
Rules:
1. Output must be valid JSON only, no explanations.
2. Keys must remain unchanged.
3. For categorical features, use ONLY allowed categories.
4. For numeric features, return integers.
5. If not inferable, keep 'None'.
6. Be strict: never invent values outside categories.
"""

In [22]:
test_queries = [
    "The house was built in 1998, has 2 garage spaces, and is located in Edwards neighborhood.",
    "Kitchen quality is excellent, basement is unfinished, and living area is 1450 sqft.",
    "Lot size is 8500, overall quality is 7, exterior quality is good."
]

In [23]:
categorical_features = [
    "ExterQual",
    "BsmtQual",
    "HeatingQC",
    "KitchenQual",
    "Neighborhood",
    "Foundation",
    "BsmtFinType1",
    "GarageType",
    "GarageFinish",
    "GarageFinish",
    "OverallQual",
    "GrLivArea"
]

In [24]:
categories = {}

for feature in categorical_features:
    categories[feature] = df[feature].sort_values().unique()

In [25]:
categories_values = """
For the following features, they can only have the specified values/categories.
If you are not able to infer the value or category of these features from the text in triple backquotes,
then keep the value None in the returned dictionary
else, fill it with the allowed categories.
It is important to keep in mind that for infering, the user might not give direct answer to fill,
so infer is important.
For the other features not specified in the following list,
they have values as numbers only and no categories or text allowed.
"""

In [30]:
def safe_parse(raw_output: str):
    """Try to parse model output into a dict safely."""
    cleaned = raw_output.strip()

    # Remove triple backticks or language hints
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`").strip()
        # Sometimes models add "json" after ```
        if cleaned.lower().startswith("json"):
            cleaned = cleaned[4:].strip()

    try:
        return json.loads(cleaned)
    except json.JSONDecodeError:
        try:
            return ast.literal_eval(cleaned)
        except Exception:
            return {"error": "Failed to parse", "raw_output": raw_output}


In [31]:
logging.basicConfig(
    filename="experiment_log.txt",
    level=logging.INFO,
    format="%(asctime)s - %(message)s"
)

def run_experiments(queries, features_dict):
    results = []
    for i, q in enumerate(queries):
        for version, prompt in [("v1", prompt_v1), ("v2", prompt_v2)]:
            full_prompt = f"{prompt}\n{features_dict}\n{categories_values}\n{categories}\n```{q}```"
            response = client.chat.completions.create(
                model="llama-3.3-70b-versatile",
                messages=[{"role": "user", "content": full_prompt}],
                temperature=0.1,
            )
            raw_output = response.choices[0].message.content
            try:
                parsed = json.loads(raw_output)
            except:
                parsed = safe_parse(raw_output)

            valid_json = isinstance(parsed, dict)
            valid_categories = all(
                (parsed.get(f) in categories[f] or parsed.get(f) == "None")
                for f in categories
                if f in parsed
            )
            validation = valid_json and valid_categories

            result = {
                "version": version,
                "input": q,
                "output": parsed,
                "validation_result": validation
            }
            results.append(result)

            # Log each result
            logging.info(
                f"Version: {version} | Input: {q} | Output: {parsed} | Validation: {validation}"
            )

    # Save results to CSV for easy comparison
    df_results = pd.DataFrame(results)
    df_results.to_csv("experiment_results.csv", index=False)

    return results

In [32]:
features = """
{
"LotArea" : "None",
"SaleCondition" : "None",
"OverallQual" : "None",
"BsmtQual" : "None",
"YearBuilt" : "None",
"ExterQual" : "None",
"GrLivArea" : "None",
"TotalBsmtSF" : "None",
"GarageCars" : "None",
"KitchenQual" : "None",
"Neighborhood" : "None",
"Condition1" : "None",
}
"""

In [33]:
results = run_experiments(test_queries, features)
df_results = pd.DataFrame(results)
print(df_results)

  version                                              input  \
0      v1  The house was built in 1998, has 2 garage spac...   
1      v2  The house was built in 1998, has 2 garage spac...   
2      v1  Kitchen quality is excellent, basement is unfi...   
3      v2  Kitchen quality is excellent, basement is unfi...   
4      v1  Lot size is 8500, overall quality is 7, exteri...   
5      v2  Lot size is 8500, overall quality is 7, exteri...   

                                              output  validation_result  
0  {'LotArea': 'None', 'SaleCondition': 'None', '...               True  
1  {'LotArea': 'None', 'SaleCondition': 'None', '...               True  
2  {'LotArea': 'None', 'SaleCondition': 'None', '...              False  
3  {'LotArea': 'None', 'SaleCondition': 'None', '...               True  
4  {'LotArea': '8500', 'SaleCondition': 'None', '...              False  
5  {'LotArea': 8500, 'SaleCondition': 'None', 'Ov...               True  


In [35]:
df_results.head(10)

,version,input,output,validation_result
0,v1,"The house was built in 1998, has 2 garage spac...","{'LotArea': 'None', 'SaleCondition': 'None', '...",True
1,v2,"The house was built in 1998, has 2 garage spac...","{'LotArea': 'None', 'SaleCondition': 'None', '...",True
2,v1,"Kitchen quality is excellent, basement is unfi...","{'LotArea': 'None', 'SaleCondition': 'None', '...",False
3,v2,"Kitchen quality is excellent, basement is unfi...","{'LotArea': 'None', 'SaleCondition': 'None', '...",True
4,v1,"Lot size is 8500, overall quality is 7, exteri...","{'LotArea': '8500', 'SaleCondition': 'None', '...",False
5,v2,"Lot size is 8500, overall quality is 7, exteri...","{'LotArea': 8500, 'SaleCondition': 'None', 'Ov...",True
